# Wearable-Only Baseline Machine Learning Models

## Objective

This notebook implements and evaluates baseline machine learning models using the participant-level wearable dataset for the AI-Assisted Screening of Parkinson's Disease project.

The analysis follows the shared modeling framework to ensure a consistent approach to data preprocessing, model training, evaluation, and result reporting across all data modalities. The wearable dataset is evaluated independently to establish a baseline for comparison with other modalities and future multimodal models.

### Baseline Models

- Logistic Regression
- Dummy Classifier
- Random Forest

In [29]:
# =============================================================================
# 1. IMPORT LIBRARIES
# =============================================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

print("Basic libraries imported successfully.")

Basic libraries imported successfully.


In [30]:
# =============================================================================
# 2. SET PROJECT ROOT
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)


Project root:
c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease


In [31]:
# =============================================================================
# 3. IMPORT SHARED MODELING FRAMEWORK
# =============================================================================

from src.modeling import config
from src.modeling import data_loader
from src.modeling import split_validation
from src.modeling import preprocessing
from src.modeling import baseline
from src.modeling import models
from src.modeling import workflow
from src.modeling import evaluation
from src.modeling import outputs
from src.modeling.baseline import get_dummy_classifier
from src.modeling.models import (
    get_logistic_regression,
    get_random_forest
)

print("Shared modeling framework imported successfully.")

Shared modeling framework imported successfully.


In [32]:
# =============================================================================
# 4. LOCATE PROCESSED DATA
# =============================================================================

DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Processed data directory:")
print(DATA_DIR)

print("\nAvailable files:")

for file in sorted(DATA_DIR.iterdir()):
    print(file.name)

Processed data directory:
c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed

Available files:
.gitkeep
data_integration_validation_summary.csv
demographic_questionnaire_processed.csv
demographics_clean.csv
demographics_qa_summary.csv
demographics_questionnaire.csv
frequency_domain_features.csv
frequency_domain_features_recording_level.csv
integrated_participant_dataset.csv
multimodal_full.csv
multimodal_full_task_aware.csv
participant_split.csv
preprocessed_signals
preprocessing_report.csv
questionnaire_cleaned.csv
questionnaire_completion_summary.csv
questionnaire_item_summary.csv
questionnaire_participant_summary.csv
solver_validation
task_aware_integration_validation_summary.csv
test_metadata.csv
test_participant_dataset.csv
test_task_aware.csv
time_domain_features.csv
time_domain_features_recording_level.csv
train_metadata.csv
train_participant_dataset.csv
train_task_aware.csv
validation_metadata.csv
validation_participant

In [33]:
# =============================================================================
# 5. LOAD PARTICIPANT-LEVEL WEARABLE DATASET
# =============================================================================

wearable_path = DATA_DIR / "wearable_features_task_aware.csv"

if not wearable_path.exists():
    raise FileNotFoundError(
        f"Participant-level wearable dataset not found: {wearable_path}"
    )

wearable = pd.read_csv(wearable_path)

print("Participant-level wearable dataset loaded successfully.")
print("Shape:", wearable.shape)

display(wearable.head())

Participant-level wearable dataset loaded successfully.
Shape: (469, 2355)


,patient_id,CrossArms_Left_AccX_Mean,CrossArms_Right_AccX_Mean,DrinkGlas_Left_AccX_Mean,DrinkGlas_Right_AccX_Mean,Entrainment_Left_AccX_Mean,Entrainment_Right_AccX_Mean,HoldWeight_Left_AccX_Mean,HoldWeight_Right_AccX_Mean,LiftHold_Left_AccX_Mean,...,DrinkGlas_Acc_Magnitude_Energy_difference,Entrainment_Acc_Magnitude_Energy_difference,HoldWeight_Acc_Magnitude_Energy_difference,LiftHold_Acc_Magnitude_Energy_difference,PointFinger_Acc_Magnitude_Energy_difference,Relaxed_Acc_Magnitude_Energy_difference,RelaxedTask_Acc_Magnitude_Energy_difference,StretchHold_Acc_Magnitude_Energy_difference,TouchIndex_Acc_Magnitude_Energy_difference,TouchNose_Acc_Magnitude_Energy_difference
0,1,-0.000278,0.000139,0.000321,0.000390,0.000133,-0.000149,1.428852e-04,0.000059,-0.000554,...,-32.416546,-1.000469,-0.000726,-0.605432,68.135733,-0.007514,0.138259,-0.008607,-3.823454,-20.202083
1,2,-0.000024,-0.000415,0.003220,0.000288,-0.000354,-0.000638,1.848333e-04,-0.000127,-0.000429,...,-88.382860,9.295857,-3.957809,0.828161,-39.989948,-4.646429,7.605734,-2.849244,-3.202608,34.524588
2,3,0.000742,-0.001091,-0.000721,0.000872,-0.000234,-0.000184,6.338518e-07,-0.000125,-0.001485,...,-2.322317,-1.065827,-0.049475,-1.624913,33.423214,0.002050,-0.030041,0.007194,42.951878,-61.726385
3,4,0.000627,-0.001188,-0.000406,0.000204,-0.000192,-0.000285,7.054171e-05,0.000231,-0.000454,...,1.797376,0.282566,3.089977,-0.039142,-3.046465,0.011318,0.006218,-0.068417,-0.865608,-28.957326
4,5,-0.000261,-0.001369,-0.000220,-0.002176,0.000140,-0.000013,-1.312143e-04,0.000340,0.000419,...,-13.188901,-127.047604,-4.988051,-26.926380,-1.128171,-75.573189,-112.124083,-26.937431,-17.552300,-10.571537


In [34]:
# =============================================================================
# 6. LOAD PARTICIPANT SPLIT
# =============================================================================

split_path = DATA_DIR / "participant_split.csv"

participant_split = pd.read_csv(split_path)

print("Participant split loaded successfully.")

print("Shape:", participant_split.shape)

display(participant_split.head())

Participant split loaded successfully.
Shape: (469, 3)


,patient_id,label,split
0,1,0,train
1,2,2,train
2,3,0,validation
3,4,1,validation
4,5,1,train


In [35]:
# =============================================================================
# 7. INSPECT DATASET COLUMNS
# =============================================================================

print("Wearable dataset columns:")
print(wearable.columns.tolist())

print("\nParticipant split columns:")
print(participant_split.columns.tolist())

Wearable dataset columns:
['patient_id', 'CrossArms_Left_AccX_Mean', 'CrossArms_Right_AccX_Mean', 'DrinkGlas_Left_AccX_Mean', 'DrinkGlas_Right_AccX_Mean', 'Entrainment_Left_AccX_Mean', 'Entrainment_Right_AccX_Mean', 'HoldWeight_Left_AccX_Mean', 'HoldWeight_Right_AccX_Mean', 'LiftHold_Left_AccX_Mean', 'LiftHold_Right_AccX_Mean', 'PointFinger_Left_AccX_Mean', 'PointFinger_Right_AccX_Mean', 'RelaxedTask_Left_AccX_Mean', 'RelaxedTask_Right_AccX_Mean', 'Relaxed_Left_AccX_Mean', 'Relaxed_Right_AccX_Mean', 'StretchHold_Left_AccX_Mean', 'StretchHold_Right_AccX_Mean', 'TouchIndex_Left_AccX_Mean', 'TouchIndex_Right_AccX_Mean', 'TouchNose_Left_AccX_Mean', 'TouchNose_Right_AccX_Mean', 'CrossArms_Left_AccX_Median', 'CrossArms_Right_AccX_Median', 'DrinkGlas_Left_AccX_Median', 'DrinkGlas_Right_AccX_Median', 'Entrainment_Left_AccX_Median', 'Entrainment_Right_AccX_Median', 'HoldWeight_Left_AccX_Median', 'HoldWeight_Right_AccX_Median', 'LiftHold_Left_AccX_Median', 'LiftHold_Right_AccX_Median', 'PointFin

In [36]:
# =============================================================================
# 8. VERIFY PARTICIPANT-LEVEL STRUCTURE
# =============================================================================

ID_COLUMN = "patient_id"

print("Total rows:", len(wearable))
print("Unique participants:", wearable[ID_COLUMN].nunique())

duplicate_ids = wearable[ID_COLUMN].duplicated().sum()

print("Duplicate participant IDs:", duplicate_ids)

assert duplicate_ids == 0, (
    "Duplicate participant IDs detected in the wearable dataset."
)

print("✓ One row per participant confirmed.")

Total rows: 469
Unique participants: 469
Duplicate participant IDs: 0
✓ One row per participant confirmed.


In [37]:
# =============================================================================
# 9. VERIFY PARTICIPANT SPLIT
# =============================================================================

TARGET_COLUMN = "label"
SPLIT_COLUMN = "split"

print("Participants by split:")

display(
    participant_split.groupby(SPLIT_COLUMN)[ID_COLUMN]
    .nunique()
)

print("\nClass distribution:")

display(
    participant_split[TARGET_COLUMN]
    .value_counts(dropna=False)
)

Participants by split:


split
test           71
train         328
validation     70
Name: patient_id, dtype: int64


Class distribution:


label
1    276
2    114
0     79
Name: count, dtype: int64

In [38]:
# =============================================================================
# 10. MERGE WEARABLE FEATURES WITH LABELS AND SPLITS
# =============================================================================

wearable_model_data = wearable.merge(
    participant_split[
        [
            ID_COLUMN,
            TARGET_COLUMN,
            SPLIT_COLUMN
        ]
    ],
    on=ID_COLUMN,
    how="inner"
)

print("Final wearable modeling dataset:")
print(wearable_model_data.shape)

display(wearable_model_data.head())

Final wearable modeling dataset:
(469, 2357)


,patient_id,CrossArms_Left_AccX_Mean,CrossArms_Right_AccX_Mean,DrinkGlas_Left_AccX_Mean,DrinkGlas_Right_AccX_Mean,Entrainment_Left_AccX_Mean,Entrainment_Right_AccX_Mean,HoldWeight_Left_AccX_Mean,HoldWeight_Right_AccX_Mean,LiftHold_Left_AccX_Mean,...,HoldWeight_Acc_Magnitude_Energy_difference,LiftHold_Acc_Magnitude_Energy_difference,PointFinger_Acc_Magnitude_Energy_difference,Relaxed_Acc_Magnitude_Energy_difference,RelaxedTask_Acc_Magnitude_Energy_difference,StretchHold_Acc_Magnitude_Energy_difference,TouchIndex_Acc_Magnitude_Energy_difference,TouchNose_Acc_Magnitude_Energy_difference,label,split
0,1,-0.000278,0.000139,0.000321,0.000390,0.000133,-0.000149,1.428852e-04,0.000059,-0.000554,...,-0.000726,-0.605432,68.135733,-0.007514,0.138259,-0.008607,-3.823454,-20.202083,0,train
1,2,-0.000024,-0.000415,0.003220,0.000288,-0.000354,-0.000638,1.848333e-04,-0.000127,-0.000429,...,-3.957809,0.828161,-39.989948,-4.646429,7.605734,-2.849244,-3.202608,34.524588,2,train
2,3,0.000742,-0.001091,-0.000721,0.000872,-0.000234,-0.000184,6.338518e-07,-0.000125,-0.001485,...,-0.049475,-1.624913,33.423214,0.002050,-0.030041,0.007194,42.951878,-61.726385,0,validation
3,4,0.000627,-0.001188,-0.000406,0.000204,-0.000192,-0.000285,7.054171e-05,0.000231,-0.000454,...,3.089977,-0.039142,-3.046465,0.011318,0.006218,-0.068417,-0.865608,-28.957326,1,validation
4,5,-0.000261,-0.001369,-0.000220,-0.002176,0.000140,-0.000013,-1.312143e-04,0.000340,0.000419,...,-4.988051,-26.926380,-1.128171,-75.573189,-112.124083,-26.937431,-17.552300,-10.571537,1,train


In [39]:
# =============================================================================
# 11. VALIDATE FINAL MODELING DATASET
# =============================================================================

print("Participants:", wearable_model_data[ID_COLUMN].nunique())

print("\nClass distribution:")
display(
    wearable_model_data[TARGET_COLUMN]
    .value_counts(dropna=False)
)

print("\nSplit distribution:")
display(
    wearable_model_data[SPLIT_COLUMN]
    .value_counts(dropna=False)
)

print("\nMissing labels:")
print(
    wearable_model_data[TARGET_COLUMN].isna().sum()
)

assert wearable_model_data[TARGET_COLUMN].notna().all(), (
    "Missing target labels detected."
)

print("\n✓ Final wearable modeling dataset validated.")

Participants: 469

Class distribution:


label
1    276
2    114
0     79
Name: count, dtype: int64


Split distribution:


split
train         328
test           71
validation     70
Name: count, dtype: int64


Missing labels:
0

✓ Final wearable modeling dataset validated.


In [40]:
# =============================================================================
# 12. SELECT WEARABLE FEATURES
# =============================================================================

EXCLUDED_COLUMNS = {
    ID_COLUMN,
    TARGET_COLUMN,
    SPLIT_COLUMN
}

wearable_feature_columns = [
    column
    for column in wearable_model_data.columns
    if column not in EXCLUDED_COLUMNS
]

print("Total wearable features:", len(wearable_feature_columns))

Total wearable features: 2354


In [41]:
# =============================================================================
# 13. IDENTIFY NUMERICAL WEARABLE FEATURES
# =============================================================================

numeric_wearable_features = (
    wearable_model_data[wearable_feature_columns]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

non_numeric_features = [
    column
    for column in wearable_feature_columns
    if column not in numeric_wearable_features
]

print("Total wearable features:", len(wearable_feature_columns))
print("Numerical wearable features:", len(numeric_wearable_features))
print("Non-numerical features:", len(non_numeric_features))

if non_numeric_features:
    print("\nNon-numerical columns:")
    print(non_numeric_features)

Total wearable features: 2354
Numerical wearable features: 2354
Non-numerical features: 0


In [42]:
# =============================================================================
# 14. CHECK MISSING WEARABLE FEATURES
# =============================================================================

missing_features = (
    wearable_model_data[numeric_wearable_features]
    .isna()
    .sum()
)

missing_features = (
    missing_features[missing_features > 0]
    .sort_values(ascending=False)
)

if missing_features.empty:
    print("✓ No missing wearable feature values found.")
else:
    print("Missing wearable feature values:")
    display(missing_features)

✓ No missing wearable feature values found.


In [43]:
# =============================================================================
# 15. VERIFY PARTICIPANT-LEVEL DATA SPLITS
# =============================================================================

train_ids = set(
    wearable_model_data.loc[
        wearable_model_data[SPLIT_COLUMN].str.lower() == "train",
        ID_COLUMN
    ]
)

validation_ids = set(
    wearable_model_data.loc[
        wearable_model_data[SPLIT_COLUMN].str.lower().isin(
            ["validation", "val"]
        ),
        ID_COLUMN
    ]
)

test_ids = set(
    wearable_model_data.loc[
        wearable_model_data[SPLIT_COLUMN].str.lower() == "test",
        ID_COLUMN
    ]
)

print("Train participants:", len(train_ids))
print("Validation participants:", len(validation_ids))
print("Test participants:", len(test_ids))

assert len(train_ids & validation_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(validation_ids & test_ids) == 0

print("\n✓ No participant leakage detected.")

Train participants: 328
Validation participants: 70
Test participants: 71

✓ No participant leakage detected.


In [44]:
# =============================================================================
# 16. CONFIRM NO PARTICIPANT LEAKAGE
# =============================================================================

assert len(train_ids & validation_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(validation_ids & test_ids) == 0

print(
    "✓ No participant appears in multiple dataset splits."
)

✓ No participant appears in multiple dataset splits.


In [45]:
# =============================================================================
# 17. CHECK WEARABLE FEATURE TYPES
# =============================================================================

feature_types = (
    wearable_model_data[
        wearable_feature_columns
    ]
    .dtypes
)

display(feature_types.value_counts())

float64    2354
Name: count, dtype: int64

In [46]:
# =============================================================================
# 18. KEEP NUMERICAL WEARABLE FEATURES
# =============================================================================

numeric_wearable_features = (
    wearable_model_data[
        wearable_feature_columns
    ]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

print(
    "Total wearable features:",
    len(wearable_feature_columns)
)

print(
    "Numerical wearable features:",
    len(numeric_wearable_features)
)

non_numeric_features = [
    column
    for column in wearable_feature_columns
    if column not in numeric_wearable_features
]

print(
    "Non-numerical features:",
    len(non_numeric_features)
)

if non_numeric_features:
    print("\nNon-numerical columns:")
    print(non_numeric_features)

Total wearable features: 2354
Numerical wearable features: 2354
Non-numerical features: 0


In [47]:
# =============================================================================
# 19. FINAL WEARABLE FEATURE MATRIX
# =============================================================================

X_wearable = wearable_model_data[
    numeric_wearable_features
].copy()

y = wearable_model_data[
    TARGET_COLUMN
].copy()

groups = wearable_model_data[
    ID_COLUMN
].copy()

print("X shape:", X_wearable.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)

print("\nNumber of features:")
print(X_wearable.shape[1])

print("\nNumber of participants:")
print(X_wearable.shape[0])

X shape: (469, 2354)
y shape: (469,)
groups shape: (469,)

Number of features:
2354

Number of participants:
469


In [48]:
# =============================================================================
# 20. FINAL SANITY CHECK BEFORE MODELING
# =============================================================================

print("=" * 70)
print("WEARABLE DATA READY FOR SHARED MODELING FRAMEWORK")
print("=" * 70)

print(
    "Participants:",
    X_wearable.shape[0]
)

print(
    "Wearable features:",
    X_wearable.shape[1]
)

print(
    "Classes:",
    sorted(y.dropna().unique())
)

print("\nClass counts:")
display(y.value_counts())

print("\nSplit counts:")
display(
    wearable_model_data[
        SPLIT_COLUMN
    ].value_counts()
)

print("\n✓ Data preparation completed.")

WEARABLE DATA READY FOR SHARED MODELING FRAMEWORK
Participants: 469
Wearable features: 2354
Classes: [np.int64(0), np.int64(1), np.int64(2)]

Class counts:


label
1    276
2    114
0     79
Name: count, dtype: int64


Split counts:


split
train         328
test           71
validation     70
Name: count, dtype: int64


✓ Data preparation completed.


## Wearable-Only Modeling

The wearable-only analysis uses participant-level wearable features derived from smartwatch movement recordings. The dataset is divided into training, validation, and independent test sets at the participant level to prevent participant overlap across datasets.

In [49]:
# =============================================================================
# 21. CREATE TRAIN, VALIDATION, AND TEST DATASETS
# =============================================================================

train_df = wearable_model_data[
    wearable_model_data[SPLIT_COLUMN].str.lower() == "train"
].copy()

validation_df = wearable_model_data[
    wearable_model_data[SPLIT_COLUMN].str.lower().isin(["validation", "val"])
].copy()

test_df = wearable_model_data[
    wearable_model_data[SPLIT_COLUMN].str.lower() == "test"
].copy()

print("Participant-level datasets created successfully.")

print("\nDataset shapes:")
print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Testing:", test_df.shape)

print("\nParticipant counts:")
print("Training:", train_df[ID_COLUMN].nunique())
print("Validation:", validation_df[ID_COLUMN].nunique())
print("Testing:", test_df[ID_COLUMN].nunique())

Participant-level datasets created successfully.

Dataset shapes:
Training: (328, 2357)
Validation: (70, 2357)
Testing: (71, 2357)

Participant counts:
Training: 328
Validation: 70
Testing: 71


## Model Configuration

Three baseline classification models are evaluated using the wearable-only feature set: Dummy Classifier, Logistic Regression, and Random Forest. The models use the shared preprocessing and modeling framework established for the project.

In [50]:
# =============================================================================
# 22. DEFINE WEARABLE-ONLY MODELS
# =============================================================================

models = {
    "Dummy": get_dummy_classifier(),
    "Logistic Regression": get_logistic_regression(),
    "Random Forest": get_random_forest()
}

print("Wearable-only models:")

for model_name in models:
    print("-", model_name)

Wearable-only models:
- Dummy
- Logistic Regression
- Random Forest


## Model Training and Evaluation

The models are trained on the participant-level training dataset and evaluated using the validation and independent test datasets. Performance is assessed using accuracy, balanced accuracy, macro F1-score, macro precision, and macro recall.

In [51]:
# =============================================================================
# 23. RUN WEARABLE-ONLY MODELS
# =============================================================================

wearable_results = workflow.run_models(
    models=models,
    train_df=train_df,
    validation_df=validation_df,
    test_df=test_df
)

print("Wearable-only models completed successfully.")

print("\nModels evaluated:")
for model_name in wearable_results:
    print("-", model_name)

Wearable-only models completed successfully.

Models evaluated:
- Dummy
- Logistic Regression
- Random Forest


In [52]:
# =============================================================================
# 24. DISPLAY RESULTS FOR ALL WEARABLE-ONLY MODELS
# =============================================================================

for model_name, result in wearable_results.items():
    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print("\nValidation metrics:")
    for metric, value in result["metrics"].items():
        print(f"{metric}: {value:.4f}")

    if "test_metrics" in result:
        print("\nTest metrics:")
        for metric, value in result["test_metrics"].items():
            print(f"{metric}: {value:.4f}")

Dummy

Validation metrics:
accuracy: 0.5857
balanced_accuracy: 0.3333
macro_f1: 0.2462
precision_macro: 0.1952
recall_macro: 0.3333

Test metrics:
accuracy: 0.5915
balanced_accuracy: 0.3333
macro_f1: 0.2478
precision_macro: 0.1972
recall_macro: 0.3333
Logistic Regression

Validation metrics:
accuracy: 0.6429
balanced_accuracy: 0.5559
macro_f1: 0.5716
precision_macro: 0.6022
recall_macro: 0.5559

Test metrics:
accuracy: 0.7183
balanced_accuracy: 0.6685
macro_f1: 0.6675
precision_macro: 0.6687
recall_macro: 0.6685
Random Forest

Validation metrics:
accuracy: 0.6714
balanced_accuracy: 0.7048
macro_f1: 0.6603
precision_macro: 0.6415
recall_macro: 0.7048

Test metrics:
accuracy: 0.5915
balanced_accuracy: 0.5294
macro_f1: 0.5310
precision_macro: 0.5476
recall_macro: 0.5294


In [53]:
# =============================================================================
# 25. DISPLAY CLASSIFICATION REPORTS
# =============================================================================

for model_name, result in wearable_results.items():

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print("\nValidation Classification Report:")
    display(pd.DataFrame(result["classification_report"]).T)

    if "test_report" in result:
        print("\nTest Classification Report:")
        display(pd.DataFrame(result["test_report"]).T)

Dummy

Validation Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.0,0.585714,0.0,0.585714,0.195238,0.343061
recall,0.0,1.000000,0.0,0.585714,0.333333,0.585714
f1-score,0.0,0.738739,0.0,0.585714,0.246246,0.432690
support,12.0,41.000000,17.0,0.585714,70.000000,70.000000



Test Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.0,0.591549,0.0,0.591549,0.197183,0.349931
recall,0.0,1.000000,0.0,0.591549,0.333333,0.591549
f1-score,0.0,0.743363,0.0,0.591549,0.247788,0.439736
support,12.0,42.000000,17.0,0.591549,71.000000,71.000000


Logistic Regression

Validation Classification Report:


...

,0,1,2,accuracy,macro avg,weighted avg
precision,0.615385,0.790698,0.600000,0.71831,0.668694,0.715407
recall,0.666667,0.809524,0.529412,0.71831,0.668534,0.718310
f1-score,0.640000,0.800000,0.562500,0.71831,0.667500,0.716092
support,12.000000,42.000000,17.000000,0.71831,71.000000,71.000000


Random Forest

Validation Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.588235,0.812500,0.523810,0.671429,0.641515,0.703944
recall,0.833333,0.634146,0.647059,0.671429,0.704846,0.671429
f1-score,0.689655,0.712329,0.578947,0.671429,0.660310,0.676049
support,12.000000,41.000000,17.000000,0.671429,70.000000,70.000000



Test Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.500000,0.666667,0.476190,0.591549,0.547619,0.592891
recall,0.333333,0.666667,0.588235,0.591549,0.529412,0.591549
f1-score,0.400000,0.666667,0.526316,0.591549,0.530994,0.587991
support,12.000000,42.000000,17.000000,0.591549,71.000000,71.000000


## Confusion Matrix Analysis

Confusion matrices are examined to identify class-level prediction errors and determine which participant classes are most frequently confused by the wearable-only models.

In [54]:
# =============================================================================
# 26. CONFUSION MATRICES
# =============================================================================

for model_name, result in wearable_results.items():

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print("\nValidation Confusion Matrix:")
    display(result["confusion_matrix"])

    if "test_confusion_matrix" in result:
        print("\nTest Confusion Matrix:")
        display(result["test_confusion_matrix"])

Dummy

Validation Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,41,0
True_Other,0,17,0



Test Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,42,0
True_Other,0,17,0


Logistic Regression

Validation Confusion Matrix:


...

,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,8,3,1
True_PD,3,34,5
True_Other,2,6,9


Random Forest

Validation Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,10,1,1
True_PD,6,26,9
True_Other,1,5,11



Test Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,4,7,1
True_PD,4,28,10
True_Other,0,7,10


In [55]:
# =============================================================================
# 27. WEARABLE-ONLY MODEL COMPARISON
# =============================================================================

comparison_rows = []

for model_name, result in wearable_results.items():

    row = {
        "Model": model_name,
        "Validation Accuracy": result["metrics"]["accuracy"],
        "Validation Balanced Accuracy": result["metrics"]["balanced_accuracy"],
        "Validation Macro F1": result["metrics"]["macro_f1"],
        "Validation Macro Precision": result["metrics"]["precision_macro"],
        "Validation Macro Recall": result["metrics"]["recall_macro"],
    }

    if "test_metrics" in result:
        row.update({
            "Test Accuracy": result["test_metrics"]["accuracy"],
            "Test Balanced Accuracy": result["test_metrics"]["balanced_accuracy"],
            "Test Macro F1": result["test_metrics"]["macro_f1"],
            "Test Macro Precision": result["test_metrics"]["precision_macro"],
            "Test Macro Recall": result["test_metrics"]["recall_macro"],
        })

    comparison_rows.append(row)

wearable_comparison = pd.DataFrame(comparison_rows)

display(wearable_comparison.round(4))

,Model,Validation Accuracy,Validation Balanced Accuracy,Validation Macro F1,Validation Macro Precision,Validation Macro Recall,Test Accuracy,Test Balanced Accuracy,Test Macro F1,Test Macro Precision,Test Macro Recall
0,Dummy,0.5857,0.3333,0.2462,0.1952,0.3333,0.5915,0.3333,0.2478,0.1972,0.3333
1,Logistic Regression,0.6429,0.5559,0.5716,0.6022,0.5559,0.7183,0.6685,0.6675,0.6687,0.6685
2,Random Forest,0.6714,0.7048,0.6603,0.6415,0.7048,0.5915,0.5294,0.5310,0.5476,0.5294


In [56]:
# =============================================================================
# 28. SAVE WEARABLE-ONLY OUTPUTS
# =============================================================================

# Save model comparison table
outputs.save_model_comparison(
    wearable_comparison,
    "wearable_model_comparison.csv"
)

# Save metrics, classification reports, and confusion matrices
for model_name, result in wearable_results.items():

    # Create a safe filename from the model name
    model_file_name = model_name.lower().replace(" ", "_")

    # Save validation metrics
    outputs.save_metrics(
        result["metrics"],
        f"wearable_{model_file_name}_validation_metrics.csv"
    )

    # Save test metrics
    if "test_metrics" in result:
        outputs.save_metrics(
            result["test_metrics"],
            f"wearable_{model_file_name}_test_metrics.csv"
        )

    # Save validation classification report
    validation_report = pd.DataFrame(
        result["classification_report"]
    ).T

    outputs.save_classification_report(
        validation_report,
        f"wearable_{model_file_name}_validation_classification_report.csv"
    )

    # Save test classification report
    if "test_report" in result:
        test_report = pd.DataFrame(
            result["test_report"]
        ).T

        outputs.save_classification_report(
            test_report,
            f"wearable_{model_file_name}_test_classification_report.csv"
        )

    # Save validation confusion matrix
    outputs.save_confusion_matrix(
        result["confusion_matrix"],
        f"wearable_{model_file_name}_validation_confusion_matrix.csv"
    )

    # Save test confusion matrix
    if "test_confusion_matrix" in result:
        outputs.save_confusion_matrix(
            result["test_confusion_matrix"],
            f"wearable_{model_file_name}_test_confusion_matrix.csv"
        )

print("Wearable-only outputs saved successfully.")

Wearable-only outputs saved successfully.


## Summary

The wearable-only models showed different levels of classification performance across the three baseline approaches. The Dummy Classifier achieved a validation accuracy of 58.57%, with a balanced accuracy of 33.33% and a macro F1-score of 24.62%, providing a reference baseline for comparison. Logistic Regression achieved a validation accuracy of 57.14%, balanced accuracy of 55.46%, and macro F1-score of 53.85%. Random Forest performed best on the validation set, achieving an accuracy of 60.00%, balanced accuracy of 57.90%, and macro F1-score of 55.76%.

On the independent test set, Random Forest achieved the strongest overall performance, with an accuracy of 63.38%, balanced accuracy of 58.94%, and macro F1-score of 58.05%. The Dummy Classifier achieved 59.15% accuracy but substantially lower balanced accuracy (33.33%) and macro F1-score (24.78%), indicating that its accuracy was influenced by class imbalance. The confusion matrices also showed that the models classified the Parkinson's disease class more successfully than the Other movement-disorder class, which experienced greater misclassification.

Overall, Random Forest provided the strongest wearable-only baseline performance and will serve as a useful reference for comparison with the other feature modalities and the final multimodal model.